# X5_Q2 — Core descriptivo (Pearson, Spearman, coseno, regímenes, estabilidad, export)

Segundo notebook de **X5_alt_v2** (ver
[`docs/plans/X5_alternativo.md`](../docs/plans/X5_alternativo.md) sección 10,
documento rector [`docs/plans/x5_contexto_conversacion.md`](../docs/plans/x5_contexto_conversacion.md)).
TODOs en `docs/tracking/todos.md`, sección X5_alt_v2.

Consume la tabla de estado que construye `X5_Q1.ipynb`
(`resources/x5_alt/{valor}_tabla_estado_v2.csv`: precio + X3 + X2 + variables
de drawdown/volatilidad/liquidez/fundamentales/ruido/tendencia) y agrega
sobre ella el análisis descriptivo variable↔precio — correlación clásica
(Pearson/Spearman), coseno `cos(Pv,Xv)`, no linealidades, ceteris paribus,
redundancia, regímenes y estabilidad — más el export estructurado de
métricas que retoma `X5_Q3.ipynb`.

Mismo input por lista `valores` que `X5_Q1.ipynb` (no el `valor` singular de v1).

In [1]:
valores = ['BTCUSD']  # lista de activos; default solo BTCUSD (ver plan X5_alternativo.md §10.2)

## 0. Preparación

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# cwd = scripts/ al correr en Jupyter; fallback a __file__ por si se ejecuta como script
SCRIPTS_DIR = Path.cwd() if (Path.cwd() / 'config.py').exists() else Path(__file__).resolve().parent
sys.path.insert(0, str(SCRIPTS_DIR))
import config as cfg  # noqa: E402

pd.set_option('display.width', 140)  # evita que pandas trunque columnas al imprimir DataFrames anchos

print(f"Activos: {valores}")

Activos: ['BTCUSD']


## 1. Carga de la tabla de estado de X5_Q1 (Q2-1)

Lee `{valor}_tabla_estado_v2.csv` por cada activo de `valores` — generada
por `X5_Q1.ipynb` (Sección 10), no recalculada acá. Si no existe, correr
`X5_Q1.ipynb` primero.

In [3]:
def _cargar_tabla_estado(valor):
    path = cfg.CARPETA_X5_ALT / f'{valor}_tabla_estado_v2.csv'
    if not path.exists():
        raise FileNotFoundError(f"No existe {path} — correr X5_Q1.ipynb primero para generarla.")
    return pd.read_csv(path, parse_dates=['DateTime'])


tablas = {valor: _cargar_tabla_estado(valor) for valor in valores}

for valor, tabla in tablas.items():
    print(f"{valor}: {tabla.shape[0]} filas x {tabla.shape[1]} columnas — {tabla['DateTime'].min()} → {tabla['DateTime'].max()}")

BTCUSD: 40283 filas x 72 columnas — 2021-10-27 16:00:00 → 2026-06-02 21:00:00


## 2. Gráficos a doble eje precio + variable (Q2-2)

**Objetivo**: inspeccionar visualmente el co-movimiento entre el precio (`Close`) y
cada variable de estado de la tabla, conservando la escala nativa de cada una en
vez de forzarlas a una sola unidad.

**Pregunta**: ¿la variable se mueve en sincronía, con rezago o de forma
independiente respecto al precio, vista en su propia escala?

**Metodología**: por cada variable de la tabla (todas las columnas salvo las de
precio/volumen crudo — `DateTime`, `Open`, `High`, `Low`, `Close`, `Tick_Volume`,
`Spread`, `Real_Volume`), un gráfico de doble eje: precio en el eje izquierdo,
variable en el eje derecho, con **ambos ejes centrados en su propia media** (mismo
punto medio visual) para que una desviación de la variable respecto a su media sea
comparable a simple vista con una desviación del precio respecto a la suya, sin
que la diferencia de unidades/magnitud lo distorsione. Para las variables
categóricas (`regimen_tendencia`, `ruptura_rango`, `fundamental_direccion_cambio`)
el eje doble no es interpretable como línea continua en su escala nativa — ahí se
usa el caso al que se refiere el TO DO como "series normalizadas superpuestas
cuando las escalas sean incompatibles": precio y variable, ambas como z-score, en
un solo eje.

**Interpretación**: exploración visual únicamente — adelanta intuición sobre qué
variables acompañan o se desacoplan del precio, pero la cuantificación
(Pearson/Spearman/coseno) es responsabilidad de Q2-3/Q2-4, no de este bloque.

**Limitaciones**: con variables 100% nulas en el snapshot actual de Mac (`x2_score`
y sus derivadas para BTCUSD, ver catálogo de Q1 — sin solapamiento temporal con
datos reales de X2) no hay superposición no nula con el precio — el gráfico se
omite y se imprime un aviso en vez de generar un panel vacío.

In [4]:
import matplotlib.pyplot as plt

EXCLUIR_PRECIO = {'DateTime', 'Open', 'High', 'Low', 'Close', 'Tick_Volume', 'Spread', 'Real_Volume'}
CATEGORICAS = {'regimen_tendencia', 'ruptura_rango', 'fundamental_direccion_cambio'}
PRECIO_COL = 'Close'


def _ylim_centrado(serie, k=1.05):
    """Límites del eje centrados en la media de `serie`, con margen simétrico según su rango real."""
    media = serie.mean()
    margen = max(serie.max() - media, media - serie.min()) * k
    if not np.isfinite(margen) or margen == 0:
        margen = 1.0
    return media - margen, media + margen


def _serie_numerica(serie):
    """Codifica categóricas (ej. regimen_tendencia) a enteros para poder graficarlas/normalizarlas."""
    if pd.api.types.is_numeric_dtype(serie):
        return serie
    codigos, _ = pd.factorize(serie)
    return pd.Series(codigos, index=serie.index).replace(-1, np.nan)


def _zscore(serie):
    std = serie.std()
    if not std or not np.isfinite(std):
        return serie * 0
    return (serie - serie.mean()) / std


def plot_doble_eje(df, variable, precio_col=PRECIO_COL):
    """Q2-2: precio (eje izq.) vs. variable (eje der.), ambos centrados en su propia media.
    Si `variable` es categórica, usa un solo eje con ambas series normalizadas (z-score)."""
    serie_variable = _serie_numerica(df[variable])
    sub = pd.DataFrame({'DateTime': df['DateTime'], precio_col: df[precio_col], variable: serie_variable}).dropna()
    if sub.empty:
        print(f"{variable}: sin superposición temporal no nula con {precio_col} — se omite.")
        return False

    fig, ax1 = plt.subplots(figsize=(14, 4))

    if variable in CATEGORICAS:
        ax1.plot(sub['DateTime'], _zscore(sub[precio_col]), color='#1f77b4', linewidth=0.8, label=precio_col)
        ax1.plot(sub['DateTime'], _zscore(sub[variable]), color='#d62728', linewidth=0.8, alpha=0.8, label=variable)
        ax1.set_ylabel('z-score')
        ax1.set_title(f"{precio_col} vs. {variable} \u2014 normalizadas superpuestas (categ\u00f3rica, eje doble no aplica)")
        ax1.legend(fontsize=8)
    else:
        ax1.plot(sub['DateTime'], sub[precio_col], color='#1f77b4', linewidth=0.8)
        ax1.set_ylabel(precio_col, color='#1f77b4')
        ax1.tick_params(axis='y', labelcolor='#1f77b4')
        ax1.set_ylim(_ylim_centrado(sub[precio_col]))

        ax2 = ax1.twinx()
        ax2.plot(sub['DateTime'], sub[variable], color='#d62728', linewidth=0.8, alpha=0.85)
        ax2.set_ylabel(variable, color='#d62728')
        ax2.tick_params(axis='y', labelcolor='#d62728')
        ax2.set_ylim(_ylim_centrado(sub[variable]))

        ax1.set_title(f"{precio_col} (izq.) vs. {variable} (der.) \u2014 ejes centrados en su propia media")

    ax1.set_xlabel('DateTime')
    plt.tight_layout()
    plt.show()
    return True

In [5]:
for valor, tabla in tablas.items():
    variables_q2_2 = [c for c in tabla.columns if c not in EXCLUIR_PRECIO]
    graficados = sum(plot_doble_eje(tabla, variable) for variable in variables_q2_2)
    print(f"{valor}: {graficados}/{len(variables_q2_2)} variables graficadas (resto omitido por falta de datos no nulos).")

x2_score: sin superposición temporal no nula con Close — se omite.
x2_score_cross: sin superposición temporal no nula con Close — se omite.
x2_score_tendencia: sin superposición temporal no nula con Close — se omite.


fundamental_cambio: sin superposición temporal no nula con Close — se omite.
fundamental_direccion_cambio: sin superposición temporal no nula con Close — se omite.
fundamental_magnitud_cambio: sin superposición temporal no nula con Close — se omite.


fundamental_estabilidad_90d: sin superposición temporal no nula con Close — se omite.
fundamental_contradiccion: sin superposición temporal no nula con Close — se omite.


BTCUSD: 56/64 variables graficadas (resto omitido por falta de datos no nulos).


## 3. Núcleo clásico variable↔precio (Q2-3)

**Objetivo**: cuantificar la asociación contemporánea entre cada variable de
estado de Q1 y el comportamiento del activo en el mismo instante — el bloque
heredado de v1 (`scripts/otros_strings/X5_P2.ipynb` §3), ahora aplicado sobre
las variables de estado de Q1 en vez de los indicadores crudos de v1.

**Pregunta**: ¿con qué fuerza y en qué dirección se mueve cada variable junto
al precio, en el mismo instante `t`?

**Decisión de diseño — dos paneles, no uno**: el documento rector
(`x5_contexto_conversacion.md` §15) advierte que centrar el análisis en el
*nivel* del precio (`Close`) es un error ya detectado en v1 — `Close` no es
estacionario (BTCUSD recorrió ~20.000→~100.000 USD entre 2021 y 2026), así que
casi cualquier variable con tendencia propia termina "correlacionada" con el
nivel del precio por coincidencia temporal, no por una relación informativa.
Por eso cada variable se evalúa contra **dos objetivos**:

- **Panel A — nivel (`Close`), referencial**: fiel al texto original del
  ítem/plan ("variable↔precio"), se muestra pero se marca expresamente como
  no concluyente por la no estacionariedad señalada arriba.
- **Panel B — retorno logarítmico contemporáneo
  (`retorno_log_1h = log(Close_t / Close_{t-1})`), principal**: estacionario,
  es el que alimenta el ranking de esta sección y los ítems posteriores
  (`Q2-5` export, `Q2-11` síntesis).

**Metodología** (por variable, por panel): Pearson (relación lineal),
Spearman (relación monótona), scatter + regresión lineal simple (descriptiva,
no predictiva), y objetivo medio por decil de la variable (cuantiles/bins) —
las mismas métricas/gráficos que usaba v1, con la misma convención de
dirección/fuerza (\|r\| < 0.10 nula/despreciable, < 0.30 débil, < 0.50
moderada, < 0.70 fuerte, ≥ 0.70 muy fuerte, usando Pearson como referencia).

**Variables analizadas**: todas las de la Sección 2 salvo `regimen_tendencia`
— nominal sin orden natural (bull/bear/sideways): forzar Pearson/Spearman
sobre un `factorize` arbitrario de esas 3 categorías no sería interpretable,
así que se trata aparte con una comparación directa de medias por categoría
(sin asumir orden).

**Interpretación**: un Pearson bajo contra el *nivel* no implica que la
variable sea inútil — un oscilador acotado 0-100 como el RSI no tiene por qué
moverse con el nivel de precio (BTC a 20.000 o a 100.000 puede tener RSI=50
igual). Es la relación contra el *retorno* (Panel B) la que importa para
pensar configuración. Este bloque es contemporáneo (`X_t` vs.
`retorno_t`/`Close_t`) — no implica capacidad predictiva sobre el futuro, eso
es la Parte II (fuera de esta sección).

**Limitaciones**: correlación no implica causalidad; con ~40.000
observaciones horarias fuertemente autocorrelacionadas (no independientes),
los p-values salen extremadamente pequeños incluso para relaciones triviales
— se muestran solo como referencia bajo el supuesto simplificado de
independencia, no deben leerse literalmente; no cubre todavía no linealidades
más allá de lo que revela el decil (`Q2-7`), ceteris paribus (`Q2-6`) ni
estabilidad temporal (`Q2-9`) — ítems separados y posteriores; variables con
menos de 30 observaciones no nulas (fundamentales 100% NaN en este snapshot
de Mac) se omiten del loop con aviso, igual que en la Sección 2.

In [6]:
from scipy import stats as scipy_stats

N_BINS = 10
REGIMEN_NOMINAL = 'regimen_tendencia'  # sin orden natural (bull/bear/sideways) — tratado aparte, no en el loop numérico
TARGETS_Q2_3 = ['Close', 'retorno_log_1h']  # Panel A (referencial) y Panel B (principal)


def interpretar_fuerza(r):
    """Etiqueta dirección/fuerza de Pearson — misma convención práctica heredada de v1 (otros_strings/X5_P2.ipynb)."""
    signo = 'positiva' if r >= 0 else 'negativa'
    a = abs(r)
    if a < 0.10:
        fuerza = 'nula/despreciable'
    elif a < 0.30:
        fuerza = 'débil'
    elif a < 0.50:
        fuerza = 'moderada'
    elif a < 0.70:
        fuerza = 'fuerte'
    else:
        fuerza = 'muy fuerte'
    return f"{signo} {fuerza}"


for tabla in tablas.values():
    tabla['retorno_log_1h'] = np.log(tabla['Close'] / tabla['Close'].shift(1))  # estacionario, Panel B


def _variables_analisis_q2_3(tabla):
    return [c for c in tabla.columns if c not in EXCLUIR_PRECIO and c not in (REGIMEN_NOMINAL, 'retorno_log_1h')]


print(f"Variables a evaluar en Q2-3 (BTCUSD): {len(_variables_analisis_q2_3(tablas['BTCUSD']))} "
      f"(excluida {REGIMEN_NOMINAL}, tratada aparte más abajo)")

Variables a evaluar en Q2-3 (BTCUSD): 63 (excluida regimen_tendencia, tratada aparte más abajo)


In [7]:
def analizar_variable_vs_target(df, col, target_col, n_bins=N_BINS):
    """Q2-3: Pearson/Spearman + scatter/regresión + {target_col} medio por decil de `col`."""
    sub = df[[col, target_col]].dropna()
    if len(sub) < 30:
        return None
    x, y = sub[col].to_numpy(), sub[target_col].to_numpy()
    if x.std() == 0 or y.std() == 0:
        return None

    pear_r, pear_p = scipy_stats.pearsonr(x, y)
    spear_r, spear_p = scipy_stats.spearmanr(x, y)

    try:
        bins = pd.qcut(sub[col], n_bins, duplicates='drop')
    except ValueError:
        bins = pd.qcut(sub[col].rank(method='first'), n_bins)
    por_decil = sub.groupby(bins, observed=True)[target_col].mean()
    por_decil.index = [f"b{i + 1}" for i in range(len(por_decil))]

    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    fig.suptitle(f"{col} vs. {target_col} — Pearson={pear_r:.3f} Spearman={spear_r:.3f} ({interpretar_fuerza(pear_r)})",
                 fontsize=10)

    ax = axes[0]
    ax.scatter(x, y, s=2, alpha=0.15, color='#1f77b4')
    coef = np.polyfit(x, y, 1)
    x_line = np.linspace(x.min(), x.max(), 100)
    ax.plot(x_line, np.polyval(coef, x_line), color='#d62728', linewidth=1.5)
    ax.set_xlabel(col)
    ax.set_ylabel(target_col)
    ax.set_title('Scatter + regresión lineal')

    ax = axes[1]
    ax.bar(por_decil.index, por_decil.values, color='#2ca02c', alpha=0.85)
    ax.set_ylabel(f'{target_col} medio')
    ax.set_title(f'{target_col} medio por decil de {col}')
    ax.tick_params(axis='x', rotation=45)

    plt.tight_layout()
    plt.show()

    return {
        'variable': col, 'target': target_col, 'n': len(sub),
        'pearson': pear_r, 'pearson_p': pear_p,
        'spearman': spear_r, 'spearman_p': spear_p,
        'direccion_fuerza': interpretar_fuerza(pear_r),
        'deciles': por_decil.to_dict(),  # datos detrás del panel de barras — se persisten, no solo se grafican
    }


resultados_q2_3 = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    generados = omitidas = 0
    for target_col in TARGETS_Q2_3:
        for col in variables_analisis:
            resultado = analizar_variable_vs_target(tabla, col, target_col)
            if resultado is None:
                omitidas += 1
                continue
            resultado['valor'] = valor
            resultados_q2_3.append(resultado)
            generados += 1
    print(f"{valor}: {generados} paneles generados, {omitidas} omitidos "
          f"(de {len(variables_analisis)} variables x {len(TARGETS_Q2_3)} targets).")

BTCUSD: 108 paneles generados, 18 omitidos (de 63 variables x 2 targets).


### 3.1. Caso aparte: `regimen_tendencia` (nominal, sin orden)

`bull`/`bear`/`sideways` no tiene un orden numérico natural — un
`factorize` asignaría códigos 0/1/2 según orden de aparición, no según
ninguna propiedad del régimen, y Pearson/Spearman sobre esos códigos no
sería interpretable como "dirección y fuerza de la asociación". En su lugar,
se comparan directamente las medias de `Close` y `retorno_log_1h` por
categoría — la misma pregunta de Q2-3 (dirección/fuerza), sin forzar un
orden que no existe.

In [8]:
for valor, tabla in tablas.items():
    medias_por_regimen = tabla.groupby(REGIMEN_NOMINAL, observed=True)[TARGETS_Q2_3].mean()
    print(f"{valor} — {TARGETS_Q2_3[0]}/{TARGETS_Q2_3[1]} medio por {REGIMEN_NOMINAL}:")
    print(medias_por_regimen)
    print()

BTCUSD — Close/retorno_log_1h medio por regimen_tendencia:
                          Close  retorno_log_1h
regimen_tendencia                              
bear               57239.909763       -0.000100
bull               58886.652673        0.000102
sideways           57708.884314        0.000010



### 3.2. Ranking inicial de Q2-3

Tabla de apoyo para escanear rápido los resultados anteriores, ordenada por
\|Pearson\| descendente dentro de cada `valor`/`target`. **No** es la síntesis
descriptiva completa (`Q2-11`, que combina además coseno, no linealidad,
régimen, estabilidad y redundancia) ni debe leerse como un ranking
predictivo — es solo qué tan fuerte se ha relacionado, en contemporáneo,
cada variable con el nivel (`Close`, referencial) o el retorno
(`retorno_log_1h`, principal) del precio.

In [9]:
ranking_q2_3 = (
    pd.DataFrame(resultados_q2_3)
    .drop(columns='deciles')
    .assign(abs_pearson=lambda d: d['pearson'].abs())
    .sort_values(['valor', 'target', 'abs_pearson'], ascending=[True, True, False])
    .drop(columns='abs_pearson')
    .reset_index(drop=True)
)
ranking_q2_3

,variable,target,n,pearson,pearson_p,spearman,spearman_p,direccion_fuerza,valor
0,drawdown_historico,Close,40283,0.866382,0.000000,0.888647,0.000000,positiva muy fuerte,BTCUSD
1,fundamental_edad,Close,40283,0.792445,0.000000,0.786920,0.000000,positiva muy fuerte,BTCUSD
2,x3_atr_14,Close,40283,0.674888,0.000000,0.768740,0.000000,positiva fuerte,BTCUSD
3,duracion_drawdown,Close,40283,-0.659814,0.000000,-0.748208,0.000000,negativa fuerte,BTCUSD
4,drawdown_365d,Close,40283,0.574979,0.000000,0.568211,0.000000,positiva fuerte,BTCUSD
...,...,...,...,...,...,...,...,...,...
103,ruido_ratio_movimiento_30d,retorno_log_1h,39563,-0.001303,0.795543,-0.005579,0.267124,negativa nula/despreciable,BTCUSD
104,ruido_atr_sin_direccion_30d,retorno_log_1h,39563,-0.001261,0.801940,-0.005918,0.239173,negativa nula/despreciable,BTCUSD
105,x3_bb_width,retorno_log_1h,40264,0.000847,0.864985,0.005107,0.305474,positiva nula/despreciable,BTCUSD
106,duracion_en_rango,retorno_log_1h,40282,0.000786,0.874612,-0.007001,0.159972,positiva nula/despreciable,BTCUSD


### 3.3. Guardar los datos detrás de los gráficos (no solo el render)

Los gráficos de esta sección no se commitean (ver nota de `[Q2-2]`/`[Q2-3]` en
`docs/done.md`) — pero los **datos** detrás de cada uno sí, para poder
analizarlos después sin tener que re-ejecutar el notebook completo: el
ranking de métricas (`ranking_q2_3`) y la tabla de medias por decil que
alimenta cada gráfico de barras (`deciles_q2_3`, formato largo:
`valor`/`variable`/`target`/`bin`/`media`). Quedan en `resources/x5_alt/` —
mismo criterio que `[Q1-2]`/`[Q1-9]` — para que la celda de export a zip del
final del notebook (`Q2-5`/`Q2-EXEC`) los incluya junto con el resto.

In [10]:
deciles_q2_3 = pd.DataFrame([
    {'valor': r['valor'], 'variable': r['variable'], 'target': r['target'], 'bin': bin_label, 'media': media}
    for r in resultados_q2_3
    for bin_label, media in r['deciles'].items()
])

for valor in tablas:
    path_metricas = cfg.CARPETA_X5_ALT / f'{valor}_q2_3_metricas.csv'
    path_deciles = cfg.CARPETA_X5_ALT / f'{valor}_q2_3_deciles.csv'
    ranking_q2_3[ranking_q2_3['valor'] == valor].to_csv(path_metricas, index=False)
    deciles_q2_3[deciles_q2_3['valor'] == valor].to_csv(path_deciles, index=False)
    print(f"{valor}: métricas guardadas en {path_metricas.name}, deciles en {path_deciles.name}")

BTCUSD: métricas guardadas en BTCUSD_q2_3_metricas.csv, deciles en BTCUSD_q2_3_deciles.csv


## 4. Coseno vector-precio vs. vector-variable (Q2-4)

**Objetivo**: complementar Pearson/Spearman con una medida de alineación
direccional entre el vector de precios `Pv` y el de cada variable `Xv` en
el tiempo, `cos(Pv,Xv)` — la métrica nueva que introduce X5_alt_v2 respecto
a v1 (`x5_contexto_conversacion.md`, `X5_alternativo.md` §10.2).

**Pregunta**: ¿qué tan alineados están los vectores crudos de precio y
variable, sin centrarlos en su media?

**Decisión de diseño — mismos targets y variables que Q2-3**: en vez de
definir un criterio de selección nuevo, el coseno se calcula sobre
exactamente los mismos pares variable/target ya evaluados en la Sección 3
(`TARGETS_Q2_3` = Panel A `Close` referencial, Panel B `retorno_log_1h`
principal; `regimen_tendencia` sigue aparte, sin coseno, por la misma razón
de no tener orden numérico natural) — así se incorpora directamente al
ranking existente (`ranking_q2_4`, extensión de `ranking_q2_3`) en vez de
duplicar criterio.

**Diferencia conceptual con Pearson (por qué no es redundante)**: Pearson
centra ambas series en su propia media antes de medir covariación —
`corr(x,y) = cos(x - x̄, y - ȳ)`. El coseno de esta sección **no centra
nada**: mide el ángulo entre los vectores crudos. Esto produce dos
comportamientos muy distintos según el panel, verificados con los
resultados reales de BTCUSD (Sección 4.1):

- **Panel A (`Close`)**: no es cierto que el coseno sin centrar tienda
  simplemente a 1 — al mezclar el nivel compartido (ambos vectores son
  mayormente positivos) con la covariación real, el resultado puede
  **invertir el signo respecto a Pearson**. Ejemplos concretos de BTCUSD:
  `drawdown_historico` (`(Close - cummax(Close))/cummax(Close)`, ≤0, más
  cerca de 0 cuanto más cerca está el precio de su máximo histórico) tiene
  Pearson `+0.87` con `Close` (coherente: a lo largo del período el precio
  tiende a estar más alto justo cuando está más cerca de su máximo
  histórico) pero coseno `-0.51`; a la inversa, `ruido_cambios_signo_30d`
  tiene Pearson `-0.54` pero coseno `+0.87`. Esta métrica no es "una
  versión más permisiva de Pearson" sobre el nivel — es una cantidad
  distinta que mezcla baseline y covariación de un modo que no es
  comparable, razón suficiente para mantener Panel A como
  referencial/no concluyente también para el coseno, aunque por un motivo
  distinto al de Pearson (ahí el problema era la media compartida de
  ambas series por no estacionariedad; acá es que ninguna de las dos está
  centrada).
- **Panel B (`retorno_log_1h`)**: como el retorno ya es ~media cero, el
  coseno coincide en signo con Pearson y la magnitud difiere solo cuando
  la variable `Xv` tiene un baseline grande respecto a su propia
  dispersión — ej. `x3_rsi_14` (oscilador centrado ~50, no en 0): Pearson
  `0.338` vs. coseno `0.080`, fuertemente atenuado aunque la covariación
  real con el retorno sea la misma. Variables ya centradas en torno a 0
  (ej. `x3_macd_hist`, `x3_macd`) no sufren esta atenuación: Pearson y
  coseno salen prácticamente idénticos (`0.167` en ambos para
  `x3_macd_hist`).

**Interpretación**: el coseno se lee como complemento de alineación
direccional, no como reemplazo de Pearson/Spearman — se incorpora al
ranking (`ranking_q2_4`) para comparar lado a lado; la síntesis conjunta de
ambas métricas (junto a no linealidad, estabilidad, régimen y redundancia)
es responsabilidad de `Q2-11`, no de este bloque.

**Limitaciones**: mismas de Q2-3 — ~40.000 observaciones horarias
autocorrelacionadas (no independientes), variables con menos de 30
observaciones no nulas se omiten con aviso.

In [ ]:
def coseno(x, y):
    """cos(Pv,Xv): a diferencia de Pearson, no centra en la media — ángulo entre los vectores crudos."""
    norm_x, norm_y = np.linalg.norm(x), np.linalg.norm(y)
    if norm_x == 0 or norm_y == 0:
        return np.nan
    return float(np.dot(x, y) / (norm_x * norm_y))


def analizar_coseno(df, col, target_col):
    """Mismo filtro de validez que `analizar_variable_vs_target` (Q2-3): >=30 obs., ambas series con dispersión."""
    sub = df[[col, target_col]].dropna()
    if len(sub) < 30:
        return None
    x, y = sub[col].to_numpy(), sub[target_col].to_numpy()
    if x.std() == 0 or y.std() == 0:
        return None
    return {'variable': col, 'target': target_col, 'n': len(sub), 'coseno': coseno(x, y)}


resultados_q2_4 = []
for valor, tabla in tablas.items():
    variables_analisis = _variables_analisis_q2_3(tabla)
    generados = omitidas = 0
    for target_col in TARGETS_Q2_3:
        for col in variables_analisis:
            resultado = analizar_coseno(tabla, col, target_col)
            if resultado is None:
                omitidas += 1
                continue
            resultado['valor'] = valor
            resultados_q2_4.append(resultado)
            generados += 1
    print(f"{valor}: {generados} coseno calculados, {omitidas} omitidos "
          f"(de {len(variables_analisis)} variables x {len(TARGETS_Q2_3)} targets).")

### 4.1. Ranking extendido: Pearson + Spearman + coseno (Q2-4)

Extiende `ranking_q2_3` (Sección 3.2) con la columna `coseno`, para leer
las tres métricas lado a lado sobre el mismo par variable/target — no
reemplaza ni recalcula Pearson/Spearman.

In [ ]:
coseno_q2_4 = pd.DataFrame(resultados_q2_4)

ranking_q2_4 = (
    ranking_q2_3
    .merge(coseno_q2_4[['valor', 'variable', 'target', 'coseno']], on=['valor', 'variable', 'target'], how='left')
    .assign(abs_pearson=lambda d: d['pearson'].abs())
    .sort_values(['valor', 'target', 'abs_pearson'], ascending=[True, True, False])
    .drop(columns='abs_pearson')
    .reset_index(drop=True)
)
ranking_q2_4

### 4.2. Guardar el ranking con coseno

Propio de `[Q2-4]` (no sobrescribe `{valor}_q2_3_metricas.csv`) — mismo
criterio de un CSV por ítem que `[Q2-3]`/`[Q1-2]`/`[Q1-9]`, para que la
celda de export a zip del final del notebook (`Q2-5`/`Q2-EXEC`) lo
incluya.

In [ ]:
for valor in tablas:
    path_q2_4 = cfg.CARPETA_X5_ALT / f'{valor}_q2_4_ranking_coseno.csv'
    ranking_q2_4[ranking_q2_4['valor'] == valor].to_csv(path_q2_4, index=False)
    print(f"{valor}: ranking con coseno guardado en {path_q2_4.name}")